In [15]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers,models
import tensorflow_io as tfio

In [31]:
def build_convnext_tiny(num_classes=182, input_shape=(None,None,3)):
    model= tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
    )
    x= layers.GlobalAveragePooling2D()(model.output)
    x= layers.Dense(512, activation='gelu')(x)
    x=layers.Dropout(0.3)(x)

    head=layers.Dense(num_classes,activation='sigmoid')
    outputs=head(x)
    
    return models.Model(inputs=model.input, outputs=outputs)
    
def load_ogg(file_name):
    audio_binary= tf.io.read_file(file_name)
    audio= tfio.audio.decode_vorbis(audio_binary) #'decode_vorbis' is the specific op for OGG files
    audio=tf.cast(np.float32)
    audio=tf.squeeze(audio,axis=-1)
    
    return audio

def data_pipeline(file_name,label):
    audio = load_ogg(file_name)
    target_length = Sample_rate *60
    audio = audio[:target_length]
    return audio, label

In [21]:
model = build_convnext_tiny()
model.summary()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4),
    loss= tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25,label_smoothing=0.1),
    metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True, curve='ROC'), #Receiver Operating Characteristic
        tf.keras.metrics.AUC(name='Precision_recall', multi_label=True, curve='PR') # Precision-Recall
            ]
)

Model: "functional_42"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_40      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_pres… │ (None, 224, 224,  │          0 │ input_layer_40[0… │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stem  │ (None, 56, 56,    │      4,896 │ convnext_tiny_pr… │
│ (Sequential)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │      4,800 │ convnext_tiny_st… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │        192 │ convnext_tiny_st… │
│ (LayerNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     37,248 │ convnext_tiny_st… │
│ (Dense)             │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│ (Activation)        │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     36,960 │ convnext_tiny_st… │
│ (Dense)             │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │         96 │ convnext_tiny_st… │
│ (LayerScale)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_144 (Add)       │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│                     │ 96)               │            │ convnext_tiny_st… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │      4,800 │ add_144[0][0]     │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │        192 │ convnext_tiny_st… │
│ (LayerNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     37,248 │ convnext_tiny_st… │
│ (Dense)             │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│ (Activation)        │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     36,960 │ convnext_tiny_st… │
│ (Dense)             │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │         96 │ convnext_tiny_st

 Total params: 28,307,222 (107.98 MB)

 Trainable params: 28,307,222 (107.98 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:

Sampling_rate= 32000
preprocessing_model= tf.keras.Sequential([
    layers.Input(shape=(None,),name="audio_input"), #allows flexibile input
    layers.MelSpectrogram(
        sampling_rate= Sampling_rate,
        num_mel_bins=128, 
        fft_length=2048,#FFT_L 2048 gives you better frequency resolution but worse time resolution
        sequence_stride=512, #horizontal step-determines how far the window
                             #moves forward before taking the next sample.
        power_to_db=True
    ),
    layers.Reshape((-1, 128, 1)),# to bypass error, -1 is time, 128 is bins, 1 is grayscale
    layers.Resizing(224,224),
    layers.Normalization()
])

In [ ]:

ds= tf.data.Dataset.from_tensor_slices((file_paths,labels))
ds= ds.map(pipeline, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.padded_batch(
    batch_size=16,
    # [None] tells TF to pad the first dimension (audio) to the max found in the batch.
    # [] tells TF the label is a scalar and needs no padding.
    padded_shapes=([None], []),
    padding_values=(0.0, 0)
)
ds=ds.prefetch(df.data.AUTOTUNE)

In [ ]:

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'convnext_bird_model.keras',
        monitor='val_Precision_recall', # Focus on PR for bird detection
        save_best_only=True,
        mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_Precision_recall',
        patience=7,
        restore_best_weights=True
    )
]

In [ ]:

history = full_model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=50,
    callbacks=callbacks
)